# 01 — EDA và chuẩn bị dữ liệu

Give Me Some Credit, 150.000 hồ sơ, mục tiêu `SeriousDlqin2yrs`.

Notebook này chỉ gọi và kiểm chứng; logic nằm trong `src/data_prep.py` để các bước sau dùng lại được đúng cùng một split và cùng một bộ cờ.

Ba quy ước tôi đặt cho bước này:

1. Không impute và không xoá dòng chỉ vì giá trị trông xấu. Ở bộ dữ liệu này missing có mang thông tin, nên tôi đánh cờ và để phần binning quyết định.
2. Split trước khi tính bất cứ tham số nào. Ranh giới bin và giá trị WOE là tham số học được, không phải bước tiền xử lý.
3. Mọi ngưỡng phải có số liệu kèm theo. Số liệu nằm trong `results/data_profile.md`.

In [1]:
import sys, sqlite3
from pathlib import Path
import numpy as np, pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'src'))
import config, data_prep

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)
print('Project root :', config.PROJECT_ROOT)
print('Seed         :', config.SEED)
print('Split ratios :', config.SPLIT_RATIOS)


Project root : D:\hoc-ai\learning-journey\credit_scoring\project6_credit_scoring
Seed         : 42
Split ratios : (0.7, 0.15, 0.15)


## 1. Nạp dữ liệu

Tên cột gốc có dấu gạch ngang nên tôi đổi hết sang snake_case ngay từ bước nạp, viết SQL ở các bước sau đỡ phải quote. Ánh xạ tên trong `config.COLUMN_MAP`.

In [2]:
raw = data_prep.load_raw()
print('shape:', raw.shape)
raw.head()


shape: (150000, 12)


,id,target,revolving_util,age,late_30_59,debt_ratio,monthly_income,open_credit_lines,late_90,real_estate_loans,late_60_89,dependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [3]:
raw.dtypes.to_frame('dtype').assign(
    n_unique=raw.nunique(),
    n_missing=raw.isna().sum(),
    pct_missing=(100*raw.isna().mean()).round(2),
)


,dtype,n_unique,n_missing,pct_missing
id,int64,150000,0,0.00
target,int64,2,0,0.00
revolving_util,float64,125728,0,0.00
age,int64,86,0,0.00
late_30_59,int64,16,0,0.00
debt_ratio,float64,114194,0,0.00
monthly_income,float64,13594,29731,19.82
open_credit_lines,int64,58,0,0.00
late_90,int64,19,0,0.00
real_estate_loans,int64,28,0,0.00


## 2. Cột `id`

`Unnamed: 0` chỉ là số thứ tự, nhưng tôi vẫn kiểm trước khi bỏ. File dữ liệu hay được sắp xếp theo thời gian hoặc theo nhãn trước khi xuất, và khi đó số thứ tự dòng thành một biến dự báo tốt mà lúc triển khai không dùng được.

In [4]:
chk = data_prep.check_index_leakage(raw)
for k, v in chk.items():
    print(f'{k}: {v}')


corr_id_target: 0.002801
bad_rate_per_index_block: [6.47, 6.47, 7.13, 6.64, 6.61, 6.45, 6.95, 6.51, 6.55, 7.07]
verdict: khong co dau hieu leakage theo thu tu dong


## 3. Bốn chỗ dữ liệu không như mô tả

Bốn quan sát dưới đây quyết định toàn bộ cách làm sạch phía sau.

### 3.1 Mã 96/98 nằm ở mức bản ghi

Nhìn từng cột thì 96 và 98 trông như giá trị đếm bất thường. Nhưng chúng xuất hiện đồng thời ở cả ba cột trên cùng một tập dòng, nên đây là mã trạng thái của cả bản ghi, không phải số lần trễ hạn.

In [5]:
dq = config.DELINQUENCY_COLS
sent_mask = raw[dq].isin(config.SENTINEL_VALUES)
print('So dong co it nhat 1 gia tri 96/98 :', int(sent_mask.any(axis=1).sum()))
print('So dong co 96/98 o CA BA cot       :', int((sent_mask.sum(axis=1) == 3).sum()))
print()
s = sent_mask.any(axis=1)
print('bad rate nhom sentinel   : %.2f%%  (n=%d)' % (100*raw.loc[s, 'target'].mean(), s.sum()))
print('bad rate nhom con lai    : %.2f%%  (n=%d)' % (100*raw.loc[~s, 'target'].mean(), (~s).sum()))


So dong co it nhat 1 gia tri 96/98 : 269
So dong co 96/98 o CA BA cot       : 269

bad rate nhom sentinel   : 54.65%  (n=269)
bad rate nhom con lai    : 6.60%  (n=149731)


269 dòng, chiếm 0,18% dữ liệu, bad rate 54,65% so với 6,60% ở phần còn lại. Tôi giữ lại và đánh cờ, để phần binning cho nhóm này thành một bin riêng. Drop hoặc impute là thay tín hiệu mạnh nhất tập bằng giá trị của nhóm trung bình.

Với 0,18% dữ liệu thì đóng góp vào Gini sẽ nhỏ. Giá trị nằm ở cách xử lý.

### 3.2 `debt_ratio` đổi đơn vị theo mẫu số

Theo định nghĩa thì đây là tỉ lệ nghĩa vụ trả nợ trên thu nhập, đáng lẽ nằm quanh 0–1. Phân vị 90 lại là 1.267. Tách theo tình trạng thu nhập thì thấy vấn đề nằm ở mẫu số.

In [6]:
grp = np.where(raw.monthly_income.isna(), 'income_MISSING',
      np.where(raw.monthly_income == 0, 'income_ZERO', 'income_>0'))
print(raw.groupby(grp).debt_ratio.describe([.5, .9, .99])[['count','50%','90%','99%','max']].round(2))
print()
print('Ti le debt_ratio > 10 trong tung nhom:')
print((raw.debt_ratio > 10).groupby(grp).mean().round(4))
print()
ok = raw.monthly_income > 0
print('KIEM GIA THUYET: debt_ratio * monthly_income (nhom co thu nhap) =')
print((raw.loc[ok,'debt_ratio'] * raw.loc[ok,'monthly_income']).describe([.5,.9]).round(1))
print('-> median ~1.650 USD/thang, dung co mot khoan tra no hang thang that.')
print('   Nhung phep nhan nay lam TREN NHOM CO THU NHAP, tuc nhom ma gia thuyet khong noi toi:')
print('   o do debt_ratio da la payment/income theo dinh nghia, nen debt_ratio*income = payment')
print('   la mot dong nhat thuc. No cho biet THANG THAM CHIEU, khong kiem duoc 31.365 dong hong.')
print('   Ket luan dung muc: gia thuyet tuong thich voi du lieu, chua co cach bac.')


                   count      50%      90%       99%       max
income_>0       118635.0     0.29     0.72      2.99   61106.5
income_MISSING   29731.0  1159.00  3785.00   8084.50  329664.0
income_ZERO       1634.0   930.00  3587.90  10504.05   60212.0

Ti le debt_ratio > 10 trong tung nhom:
income_>0         0.0056
income_MISSING    0.9004
income_ZERO       0.8856
Name: debt_ratio, dtype: float64

KIEM GIA THUYET: debt_ratio * monthly_income (nhom co thu nhap) =
count    118635.0
mean       2146.9
std        4218.0
min           0.0
50%        1649.7
90%        4396.5
max      478450.6
dtype: float64
-> median ~1.650 USD/thang, dung co mot khoan tra no hang thang that.
   Nhung phep nhan nay lam TREN NHOM CO THU NHAP, tuc nhom ma gia thuyet khong noi toi:
   o do debt_ratio da la payment/income theo dinh nghia, nen debt_ratio*income = payment
   la mot dong nhat thuc. No cho biet THANG THAM CHIEU, khong kiem duoc 31.365 dong hong.
   Ket luan dung muc: gia thuyet tuong thich voi du lie

Nhân ngược lại trên nhóm có thu nhập cho median khoảng 1.650 USD/tháng, đúng cỡ một khoản trả nợ hàng tháng. Vậy khi thu nhập thiếu hoặc bằng 0, cột này chuyển thành số tiền tuyệt đối.

Tôi tách `debt_ratio_valid` chỉ giữ giá trị khi thu nhập dương, kèm cờ `flag_debt_ratio_invalid`. Nếu bin thẳng cột gốc thì các bin cao sẽ toàn nhóm thiếu thu nhập, tức biến đo "khách có khai thu nhập không" thay vì đo mức nợ.

### 3.3 Missing mang thông tin, và theo chiều ngược trực giác

In [7]:
for col in ['monthly_income', 'dependents']:
    m = raw[col].isna()
    print(f'{col}:')
    print('   missing     : %.2f%%  -> bad rate %.2f%%' % (100*m.mean(), 100*raw.loc[m,'target'].mean()))
    print('   co gia tri  : %.2f%%  -> bad rate %.2f%%' % (100*(~m).mean(), 100*raw.loc[~m,'target'].mean()))
z = raw.monthly_income == 0
print('monthly_income == 0 : n=%d -> bad rate %.2f%%' % (z.sum(), 100*raw.loc[z,'target'].mean()))
print('\nBase rate toan tap  : %.3f%%' % (100*raw.target.mean()))


monthly_income:
   missing     : 19.82%  -> bad rate 5.61%
   co gia tri  : 80.18%  -> bad rate 6.95%
dependents:
   missing     : 2.62%  -> bad rate 4.56%
   co gia tri  : 97.38%  -> bad rate 6.74%
monthly_income == 0 : n=1634 -> bad rate 4.04%

Base rate toan tap  : 6.684%


Người không khai thu nhập lại an toàn hơn người có khai. Cơ chế missing ở đây không phải MCAR, nên impute median là làm nhoè mất chênh lệch này. Cách xử lý là để missing thành một bin riêng và để WOE của nó do dữ liệu quyết định.

Ô dưới kiểm một giả thuyết cho chiều lệch đó.

In [8]:
# Kiem gia thuyet 'nhom huu tri' - viec con de mo #4 trong data_profile.md
m = raw.monthly_income.isna()
print(raw.groupby(m).age.describe([.25,.5,.75]).round(1)[['count','25%','50%','75%','max']])
print()
print('Ti le tuoi >= 65:')
print((raw.age >= 65).groupby(m).mean().round(4))


                   count   25%   50%   75%    max
monthly_income                                   
False           120269.0  40.0  51.0  61.0  103.0
True             29731.0  46.0  57.0  67.0  109.0

Ti le tuoi >= 65:
monthly_income
False    0.1846
True     0.3026
Name: age, dtype: float64


### 3.4 Ngưỡng rác của `revolving_util`

Tôi chọn ngưỡng theo nghĩa của biến, không dò theo target. `revolving_util` là dư nợ chia hạn mức, giá trị 10 nghĩa là dùng 1000% hạn mức. Dò ngưỡng theo target là để nhãn dẫn dắt một quyết định tiền xử lý. Chọn xong mới nhìn dữ liệu xem nhóm bị gắn cờ cư xử ra sao.

In [9]:
bands = [1.0, 1.2, 1.5, 2, 3, 5, 10, 20, 50, 100, 1e9]
b = pd.cut(raw.revolving_util, bands)
out = raw.groupby(b, observed=True).target.agg(n='size', bad='sum')
out['bad_rate_pct'] = (100*out.bad/out.n).round(2)
print('base rate = %.3f%%' % (100*raw.target.mean()))
print(out.to_string())


base rate = 6.684%
                          n  bad  bad_rate_pct
revolving_util                                
(1.0, 1.2]             2283  872         38.20
(1.2, 1.5]              438  209         47.72
(1.5, 2.0]              229  102         44.54
(2.0, 3.0]               79   25         31.65
(3.0, 5.0]               38    9         23.68
(5.0, 10.0]              13    3         23.08
(10.0, 20.0]              7    4         57.14
(20.0, 50.0]              4    1         25.00
(50.0, 100.0]             7    1         14.29
(100.0, 1000000000.0]   223   11          4.93


Nhóm trên 100 gồm 223 dòng, giá trị lớn nhất 50.708, bad rate 4,93%. Thấp hơn base rate, tức nhóm này cư xử như một mẫu ngẫu nhiên và không mang thông tin rủi ro. Ngược lại nhóm 1,0–2,0 có bad rate 38–48%, là rủi ro thật của người vượt hạn mức nên phải giữ nguyên.

Cờ `flag_util_implausible` đặt ở ngưỡng 10 để tách nhóm rác khỏi bin cao nhất khi binning.

Ngưỡng `UTIL_IMPLAUSIBLE = 10` chọn theo nghĩa của biến: 10 nghĩa là dùng 1000% hạn mức. Tôi không dò ngưỡng theo target, vì làm vậy là để nhãn quyết định một bước tiền xử lý.

Nhưng cần một cách phân biệt lỗi dữ liệu với rủi ro thật mà không dùng nhãn. Utilization thật là thương của hai số tiền nên có nhiều chữ số thập phân; một giá trị nguyên lớn thì không phải kết quả của phép chia đó, nó là một con số khác bị ghi vào cột, đúng cơ chế đã thấy ở `DebtRatio`.

In [10]:
# Nguong UTIL_IMPLAUSIBLE: gia tri nao la loi du lieu, quyet dinh KHONG dua vao nhan.
# Utilization that la thuong cua hai so tien nen co nhieu chu so thap phan.
# Mot gia tri nguyen lon khong phai ket qua cua phep chia do.
u = raw.revolving_util
nguyen = np.isclose(u % 1, 0)
print(f'{"dai":12s}{"n":>7s}{"%nguyen":>9s}{"bad|nguyen":>12s}{"bad|thap phan":>15s}')
for lo, hi, nhan in [(1, 2, '(1,2]'), (2, 10, '(2,10]'), (10, 100, '(10,100]'), (100, 1e18, '>100')]:
    m = (u > lo) & (u <= hi)
    bn = 100*raw.target[m & nguyen].mean() if (m & nguyen).sum() else float('nan')
    bt = 100*raw.target[m & ~nguyen].mean() if (m & ~nguyen).sum() else float('nan')
    print(f'{nhan:12s}{m.sum():7d}{100*nguyen[m].mean():8.1f}%{bn:11.2f}%{bt:14.2f}%')
inval = raw.monthly_income.isna() | (raw.monthly_income == 0)
m10 = (u > 10) & (u <= 100)
print(f'\n%ban ghi co thu nhap hong: toan tap {100*inval.mean():.2f}%'
      f' | (10,100] nguyen {100*inval[m10 & nguyen].mean():.2f}%'
      f' | (10,100] thap phan {100*inval[m10 & ~nguyen].mean():.2f}%')

dai               n  %nguyen  bad|nguyen  bad|thap phan
(1,2]          2950     0.0%        nan%         40.10%
(2,10]          130     1.5%       0.00%         28.91%
(10,100]         18    61.1%       0.00%         85.71%
>100            223    99.6%       4.50%        100.00%

%ban ghi co thu nhap hong: toan tap 20.91% | (10,100] nguyen 54.55% | (10,100] thap phan 0.00%


Chữ ký này tách rất sạch. Nhóm trên 100 gần như toàn số nguyên (99,6%) và bad rate của chúng thấp hơn base rate. Nhóm (10, 100] là hỗn hợp: 11 dòng nguyên có bad rate 0%, còn 7 dòng có phần thập phân thì 6 dòng là bad. Và hơn một nửa số dòng nguyên trong nhóm đó có thu nhập hỏng, so với 0% ở nhóm thập phân.

Hệ quả cho ngưỡng: hạ xuống 10 là đúng hướng vì nó chặn được cả nhóm lỗi, cái giá là gắn cờ nhầm 7 dòng thật. Nâng lên 100 thì ngược lại, thả 11 dòng lỗi vào model. Bảy dòng trên 150.000 nên tôi giữ 10 và ghi lại đây; tiêu chí đúng hơn là dấu hiệu, không phải một ngưỡng độ lớn.

## 4. Dòng trùng lặp

Phản xạ thông thường là drop. Ở đây tôi giữ, vì mấy con số dưới đây.

In [11]:
feat = [c for c in raw.columns if c not in ('id','target')]
dup = raw[feat].duplicated(keep=False)
g = raw[dup].groupby(feat, dropna=False).target.agg(n='size', nlab='nunique')
print('So dong trung feature      :', int(dup.sum()), 'trong', len(g), 'nhom')
nmt = int((g.nlab > 1).sum())
print('So nhom co NHAN MAU THUAN  :', nmt, '<- rieng', nmt, 'nhom nay chac chan la NGUOI KHAC NHAU')
# Nhung 37 nhom chi chua 145/1.000 dong, va con so 37 con THAP hon ky vong:
p_ = raw[dup].target.mean()
exp = sum(1 - (p_**n + (1-p_)**n) for n in g.n)
var = sum((1 - (p_**n + (1-p_)**n)) * (p_**n + (1-p_)**n) for n in g.n)
print(f'   ky vong neu TAT CA la nguoi doc lap: {exp:.1f} nhom (sd {var**0.5:.1f})'
      f'  -> quan sat {nmt} nam o z = {(nmt-exp)/var**0.5:+.2f}')
print(f'   tuc bang chung nay dut diem cho {int(g.n[g.nlab>1].sum())} dong, khong noi gi ve phan con lai')
print()
d = raw[dup]
print('Cac dong trung la HO SO RONG (thin file):')
for c in ['revolving_util','debt_ratio','open_credit_lines']:
    print(f'   {c:20s} == 0 : {100*(d[c]==0).mean():6.2f}%   (toan tap {100*(raw[c]==0).mean():6.2f}%)')
print(f'   {"monthly_income":20s} missing: {100*d.monthly_income.isna().mean():6.2f}%   (toan tap {100*raw.monthly_income.isna().mean():6.2f}%)')
print(f'   {"tuoi":20s} <= 25   : {100*(d.age<=25).mean():6.2f}%   (toan tap {100*(raw.age<=25).mean():6.2f}%)')
print(f'   phan bo tuoi nhom trung LUONG DINH: p25={d.age.quantile(.25):.0f} p50={d.age.median():.0f} p75={d.age.quantile(.75):.0f}'
      f'  -> median 52 giong toan tap nhung hai dinh o hai dau')


So dong trung feature      : 1000 trong 354 nhom
So nhom co NHAN MAU THUAN  : 37 <- rieng 37 nhom nay chac chan la NGUOI KHAC NHAU
   ky vong neu TAT CA la nguoi doc lap: 55.1 nhom (sd 6.7)  -> quan sat 37 nam o z = -2.69
   tuc bang chung nay dut diem cho 145 dong, khong noi gi ve phan con lai

Cac dong trung la HO SO RONG (thin file):
   revolving_util       == 0 :  49.80%   (toan tap   7.25%)
   debt_ratio           == 0 :  98.10%   (toan tap   2.74%)
   open_credit_lines    == 0 :  30.10%   (toan tap   1.26%)
   monthly_income       missing:  85.00%   (toan tap  19.82%)
   tuoi                 <= 25   :  27.00%   (toan tap   2.02%)
   phan bo tuoi nhom trung LUONG DINH: p25=25 p50=52 p75=70  -> median 52 giong toan tap nhung hai dinh o hai dau


Các dòng trùng gần như toàn bộ là hồ sơ thin file: nhiều người 22–23 tuổi, chưa có thu nhập ghi nhận, không nợ, một hoặc không có hạn mức nào. Hai người như vậy trùng khớp mười biến vì gần như không có gì để phân biệt họ.

37 nhóm có nhãn mâu thuẫn, cùng hệt mười biến nhưng một người default còn người kia không. Nếu là bản sao của cùng một hồ sơ thì nhãn phải giống nhau, nên đây là những người khác nhau.

Xoá đi là mất 646 người thật, và mất lệch hẳn về phía nhóm thin file.

## 5. Chạy pipeline và split

Split phân tầng theo nhãn, seed cố định. Phân tầng để bad rate ba tập gần bằng nhau, tránh chuyện một phần chênh lệch Gini giữa các tập đến từ tỉ lệ dương khác nhau, không phải từ model.

Đổi lại, phân tầng làm tập OOT này khác một tập out-of-time thật. Một tập out-of-time thật lấy từ giai đoạn sau, và việc bad rate giai đoạn sau lệch đi chính là thứ cần phát hiện. Hệ quả phải nhớ khi đọc kết quả ở các bước sau: PSI, bad rate và calibration trên OOT đều sẽ đẹp theo thiết kế, không phải theo năng lực model.

In [12]:
df, report = data_prep.build(verbose=False)
for k, v in report.items():
    if k not in ('index_check','split_summary'):
        print(f'{k}: {v}')
print()
pd.DataFrame(report['split_summary']).T


n_rows_raw: 150000
dropped_age_le_0: 1
duplicate_feature_rows_kept: 1000
n_rows_final: 149999
bad_rate_overall: 6.684



,n,bad,bad_rate_pct
oot,22500.0,1504.0,6.6844
test,22500.0,1504.0,6.6844
train,104999.0,7018.0,6.6839


Chỉ một dòng bị bỏ: `age = 0`, id 65696, mọi cột khác bình thường. Cách xử lý khác hẳn 646 dòng trùng ở trên, và lý do cũng khác: trùng lặp là hiện tượng có thật của dữ liệu thưa, còn `age = 0` là lỗi nhập liệu.

In [13]:
df[['id','split','target','revolving_util','age','debt_ratio_valid','flag_sentinel',
    'flag_income_missing','flag_util_implausible','flag_debt_ratio_invalid']].head(10)


,id,split,target,revolving_util,age,debt_ratio_valid,flag_sentinel,flag_income_missing,flag_util_implausible,flag_debt_ratio_invalid
0,1,train,1,0.766127,45,0.802982,0,0,0,0
1,2,oot,0,0.957151,40,0.121876,0,0,0,0
2,3,test,0,0.658180,38,0.085113,0,0,0,0
3,4,train,0,0.233810,30,0.036050,0,0,0,0
4,5,train,0,0.907239,49,0.024926,0,0,0,0
5,6,oot,0,0.213179,74,0.375607,0,0,0,0
6,7,train,0,0.305682,57,NaN,0,1,0,1
7,8,test,0,0.754464,39,0.209940,0,0,0,0
8,9,train,0,0.116951,27,NaN,0,1,0,1
9,10,train,0,0.189169,57,0.606291,0,0,0,0


## 6. Nạp vào SQLite

Một bảng kèm cột `split`, không tách ba bảng riêng. Ở bước tính WOE, lọc là `WHERE split = 'train'` và áp là `LEFT JOIN` bảng tra vào toàn bộ bảng. Không có tập nào tồn tại riêng để có thể vô tình fit lên.

In [14]:
data_prep.write_sqlite(df)
print('Da ghi:', config.DB_PATH, '|', config.DB_PATH.stat().st_size/1e6, 'MB')

con = sqlite3.connect(config.DB_PATH)
print()
print(pd.read_sql(f'PRAGMA table_info({config.TABLE})', con)[['cid','name','type']].to_string(index=False))


Da ghi: D:\hoc-ai\learning-journey\credit_scoring\project6_credit_scoring\data\credit.db | 318.971904 MB

 cid                    name    type
   0                      id INTEGER
   1                   split    TEXT
   2                  target INTEGER
   3          revolving_util    REAL
   4                     age INTEGER
   5              late_30_59 INTEGER
   6              debt_ratio    REAL
   7          monthly_income    REAL
   8       open_credit_lines INTEGER
   9                 late_90 INTEGER
  10       real_estate_loans INTEGER
  11              late_60_89 INTEGER
  12              dependents    REAL
  13           flag_sentinel INTEGER
  14     flag_income_missing INTEGER
  15        flag_income_zero INTEGER
  16 flag_dependents_missing INTEGER
  17   flag_util_implausible INTEGER
  18 flag_debt_ratio_invalid INTEGER
  19        debt_ratio_valid    REAL


## 7. Đối chiếu với `results/data_profile.md`

`data_profile.md` ghi kết quả khảo sát trên cả 150.000 dòng, trước khi split. Ô dưới đối chiếu lại từ DB và assert, để nếu sau này pipeline đổi mà quên thì notebook đỏ ngay, không trôi qua.

In [15]:
q = '''
SELECT split, COUNT(*) n, SUM(target) bad,
       ROUND(100.0*SUM(target)/COUNT(*), 4) bad_rate_pct,
       SUM(flag_sentinel) sentinel, SUM(flag_income_missing) inc_missing,
       SUM(flag_income_zero) inc_zero, SUM(flag_dependents_missing) dep_missing,
       SUM(flag_util_implausible) util_bad, SUM(flag_debt_ratio_invalid) dr_invalid
FROM applications GROUP BY split ORDER BY split
'''
per_split = pd.read_sql(q, con)
print(per_split.to_string(index=False))

tot = per_split.drop(columns=['split','bad_rate_pct']).sum()
expected = dict(n=149_999, bad=10_026, sentinel=269, inc_missing=29_731,
                inc_zero=1_634, dep_missing=3_924, util_bad=241, dr_invalid=31_365)
print()
for k, v in expected.items():
    got = int(tot[k]); ok = 'OK ' if got == v else 'SAI'
    print(f'  [{ok}] {k:12s} DB = {got:>7,}   ky vong = {v:>7,}')
    assert got == v, f'{k}: {got} != {v}'

# bad rate ba tap phai gan nhu bang nhau (phan tang)
assert per_split.bad_rate_pct.max() - per_split.bad_rate_pct.min() < 0.01
print('\n[OK ] bad rate ba tap chenh nhau < 0,01 diem phan tram -> phan tang dung')

# OOT phai co ~1.504 ca duong -> KTC 95% cua Gini = +-0,028 (bang sai so trong notes)
n_oot_bad = int(per_split.loc[per_split.split=='oot','bad'].iloc[0])
print(f'[OK ] OOT-proxy co {n_oot_bad} ca duong -> KTC 95% cua Gini = +-0,028')
print('     => moi chenh lech duoi ~0,03 Gini tren OOT phai coi la CHUA KET LUAN DUOC')


split      n  bad  bad_rate_pct  sentinel  inc_missing  inc_zero  dep_missing  util_bad  dr_invalid
  oot  22500 1504        6.6844        44         4490       234          583        32        4724
 test  22500 1504        6.6844        32         4434       230          588        32        4664
train 104999 7018        6.6839       193        20807      1170         2753       177       21977

  [OK ] n            DB = 149,999   ky vong = 149,999
  [OK ] bad          DB =  10,026   ky vong =  10,026
  [OK ] sentinel     DB =     269   ky vong =     269
  [OK ] inc_missing  DB =  29,731   ky vong =  29,731
  [OK ] inc_zero     DB =   1,634   ky vong =   1,634
  [OK ] dep_missing  DB =   3,924   ky vong =   3,924
  [OK ] util_bad     DB =     241   ky vong =     241
  [OK ] dr_invalid   DB =  31,365   ky vong =  31,365

[OK ] bad rate ba tap chenh nhau < 0,01 diem phan tram -> phan tang dung
[OK ] OOT-proxy co 1504 ca duong -> KTC 95% cua Gini = +-0,028
     => moi chenh lech duoi ~0

In [16]:
# Tai lap: chay lai build() phai cho DUNG cung mot split
df2, _ = data_prep.build(verbose=False)
assert (df2.split.values == df.split.values).all()
print('[OK ] build() tai lap duoc hoan toan voi seed =', config.SEED)
con.close()


[OK ] build() tai lap duoc hoan toan voi seed = 42


## Tình trạng sau bước này

| Việc | |
|---|---|
| Nạp CSV, đổi tên cột | xong |
| Kiểm `id` không leak rồi mới bỏ | corr 0,0028; bad rate phẳng qua 10 khối |
| Mã 96/98 thành cờ, giữ nguyên giá trị | xong |
| Missing thành cờ, không impute | xong |
| `debt_ratio` tách hai chế độ | `debt_ratio_valid` + cờ |
| Outlier `revolving_util` thành cờ, không cắt | xong |
| Dòng trùng lặp | giữ |
| `age = 0` | bỏ 1 dòng |
| Split 70/15/15 phân tầng, seed 42 | xong |
| SQLite, một bảng + cột `split` | xong |
| Đối chiếu `data_profile.md` | khớp toàn bộ |

Bước tiếp theo: binning bằng SQL, `NTILE` cho biến liên tục và `CASE` cho biến đếm cùng các mã đặc biệt, rồi tính WOE/IV chỉ trên `split = 'train'` và JOIN áp ngược vào toàn bộ bảng.